In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:

df = pd.read_csv("../data/customer_segments.csv")

In [4]:
df.head()

,CustomerID,recency,frequency,monetary,monetary_log,frequency_log,Cluster,Segment
0,12347,2,7,4310.00,8.368925,2.079442,1,Champions
1,12348,75,4,1437.24,7.271175,1.609438,1,Champions
2,12349,19,1,1457.55,7.285198,0.693147,0,normal customers
3,12350,310,1,294.40,5.688330,0.693147,2,Lost
4,12352,36,7,1385.74,7.234711,2.079442,1,Champions


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4335 entries, 0 to 4334
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CustomerID     4335 non-null   int64  
 1   recency        4335 non-null   int64  
 2   frequency      4335 non-null   int64  
 3   monetary       4335 non-null   float64
 4   monetary_log   4335 non-null   float64
 5   frequency_log  4335 non-null   float64
 6   Cluster        4335 non-null   int64  
 7   Segment        4335 non-null   object 
dtypes: float64(3), int64(4), object(1)
memory usage: 271.1+ KB


In [9]:
df["churn"] =(df["recency"]>df["recency"].median()).astype(int)

In [10]:
df['churn'].value_counts()

churn
0    2183
1    2152
Name: count, dtype: int64

there is no class imbalance 

In [11]:
df.head()

,CustomerID,recency,frequency,monetary,monetary_log,frequency_log,Cluster,Segment,churn
0,12347,2,7,4310.00,8.368925,2.079442,1,Champions,0
1,12348,75,4,1437.24,7.271175,1.609438,1,Champions,1
2,12349,19,1,1457.55,7.285198,0.693147,0,normal customers,0
3,12350,310,1,294.40,5.688330,0.693147,2,Lost,1
4,12352,36,7,1385.74,7.234711,2.079442,1,Champions,0


In [13]:
x = df[['frequency_log', 'monetary_log']]
y = df['churn']

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [15]:
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(3468, 2) (867, 2)
churn
0    0.50346
1    0.49654
Name: proportion, dtype: float64
churn
0    0.504037
1    0.495963
Name: proportion, dtype: float64


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
log_reg = LogisticRegression(random_state=42)
cv_scores = cross_val_score(
    log_reg,
    X_train,
    y_train,
    cv=5,
    scoring='f1'
)
print("result of all folds:", cv_scores)
print("average of F1-score:", cv_scores.mean())
print("std : ", cv_scores.std())

result of all folds: [0.71608392 0.72394366 0.70292887 0.71900826 0.70752089]
average of F1-score: 0.7138971208352693
std :  0.00765242018800229


In [18]:
# قيم C اللي هنجربها
C_values = [0.01, 0.1, 1, 10, 100]

results = []

for C in C_values:
    for penalty in ['l1', 'l2']:
        model = LogisticRegression(
            C=C,
            penalty=penalty,
            solver='liblinear',
            random_state=42
        )
        
        scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
        
        results.append({
            'C': C,
            'penalty': penalty,
            'mean_f1': scores.mean(),
            'std_f1': scores.std()
        })

# تحويل النتايج لجدول عشان يبقى واضح للمقارنة
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('mean_f1', ascending=False)
print(results_df)

c:\Users\i seven\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\i seven\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\i seven\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its defa

        C penalty   mean_f1    std_f1
5    1.00      l2  0.714257  0.007382
2    0.10      l1  0.714060  0.007523
8  100.00      l1  0.713897  0.007652
7   10.00      l2  0.713897  0.007652
4    1.00      l1  0.713897  0.007652
6   10.00      l1  0.713897  0.007652
9  100.00      l2  0.713897  0.007652
3    0.10      l2  0.713626  0.008725
1    0.01      l2  0.708017  0.018143
0    0.01      l1  0.688229  0.021625


c:\Users\i seven\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\i seven\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\i seven\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its defa

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# الموديل النهائي بأفضل إعدادات
final_model = LogisticRegression(
    C=1.0,
    penalty='l2',
    solver='liblinear',
    random_state=42
)

# التدريب على كل بيانات الـ train
final_model.fit(X_train, y_train)

# التوقع على الـ test المعزول
y_pred = final_model.predict(X_test)

# التقييم
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.6874279123414071
Precision: 0.6687898089171974
Recall: 0.7325581395348837
F1-score: 0.6992230854605993

Confusion Matrix:
[[281 156]
 [115 315]]


c:\Users\i seven\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10]
}
rf_model = RandomForestClassifier(random_state=42)
rf_grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=rf_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
rf_grid_search.fit(X_train, y_train)
print("best parameters (Random Forest):", rf_grid_search.best_params_)
print("best F1-score (CV):", rf_grid_search.best_score_)

best parameters (Random Forest): {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
best F1-score (CV): 0.7235612126943682


In [23]:
from xgboost import XGBClassifier
xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=xgb_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
xgb_grid_search.fit(X_train, y_train)
print("best parameters (XGBoost):", xgb_grid_search.best_params_)
print("best F1-score (CV):", xgb_grid_search.best_score_)

best parameters (XGBoost): {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300}
best F1-score (CV): 0.7236561930490395


In [24]:
best_rf = rf_grid_search.best_estimator_
best_xgb = xgb_grid_search.best_estimator_
rf_pred = best_rf.predict(X_test)
xgb_pred = best_xgb.predict(X_test)
def evaluate_model(name, y_true, y_pred):
    print(f"===== {name} =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1-score:", f1_score(y_true, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print()
evaluate_model("Random Forest", y_test, rf_pred)
evaluate_model("XGBoost", y_test, xgb_pred)

===== Random Forest =====
Accuracy: 0.6828143021914648
Precision: 0.655310621242485
Recall: 0.7604651162790698
F1-score: 0.7039827771797632
Confusion Matrix:
[[265 172]
 [103 327]]

===== XGBoost =====
Accuracy: 0.6908881199538639
Precision: 0.6626506024096386
Recall: 0.7674418604651163
F1-score: 0.7112068965517241
Confusion Matrix:
[[269 168]
 [100 330]]



In [25]:
import joblib

joblib.dump(best_xgb, '../models/churn_model.pkl')

['../models/churn_model.pkl']

In [26]:
df.to_csv("../data/customer_segments_with_churn.csv", index=False)